# HYCOM EDA — Gulf of Maine, Aug 1–3 2019

This notebook explores the HYCOM GLBv0.08 ocean reanalysis tile that was fetched during project setup.

**What you're looking at:** 3 days of 3-hourly ocean data over a small patch of the Gulf of Maine (~43–44°N, 70.5–69.5°W).

**Four variables:**
- `water_temp` — ocean potential temperature [°C]
- `salinity` — practical salinity [PSU]
- `water_u` — zonal (east-west) current [m/s]
- `water_v` — meridional (north-south) current [m/s]

**11 depth levels [m]:** 0, 5, 10, 20, 30, 50, 75, 100, 150, 200, 300  
These are already interpolated from HYCOM's native hybrid coordinate onto standard depths by `HYCOMLoader`.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Run from the repo root so relative paths resolve
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

ZARR_PATH = 'data/processed/hycom_2019-08-01_2019-08-03.zarr'
ds = xr.open_zarr(ZARR_PATH).load()
ds

## 1. Dataset structure

Before plotting anything, understand the shape of what you have.

In [ ]:
print('=== Dimensions ===')
for dim, size in ds.dims.items():
    print(f'  {dim:10s}: {size}')

print()
print('=== Time axis (3-hourly) ===')
print(ds.time.values)

print()
print('=== Depth levels [m] ===')
print(ds.depth.values)

print()
print('=== Spatial extent ===')
print(f'  lat: {ds.lat.values[0]:.3f} → {ds.lat.values[-1]:.3f}  ({len(ds.lat)} points)')
print(f'  lon: {ds.lon.values[0]:.3f} → {ds.lon.values[-1]:.3f}  ({len(ds.lon)} points)')

## 2. Surface temperature map

`depth=0` is the ocean surface (0 m). This is the variable used as the MHW detection input.

In [ ]:
# Pick one snapshot: Aug 2 at noon
t_idx = 12  # 2019-08-02T12:00
sst = ds['water_temp'].isel(depth=0, time=t_idx)

fig, ax = plt.subplots(figsize=(7, 5))
img = sst.plot(
    ax=ax, cmap='RdYlBu_r',
    cbar_kwargs={'label': 'Temperature [°C]'},
)
ax.set_title(f'Sea Surface Temperature — {str(ds.time.values[t_idx])[:16]}')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Mark the 18°C threshold (Gulf of Maine salmon stress onset)
sst.plot.contour(ax=ax, levels=[18.0], colors='black', linewidths=1.5)
ax.clabel(ax.contour(ds.lon.values, ds.lat.values, sst.values, levels=[18.0]), fmt='18°C')
plt.tight_layout()
plt.show()

print(f'SST range: {float(sst.min()):.2f}°C → {float(sst.max()):.2f}°C')
print(f'Mean SST: {float(sst.mean()):.2f}°C')

## 3. All four variables at the surface — side by side

One snapshot, one depth level. Gets you a feel for the spatial structure of each variable.

In [ ]:
variables = {
    'water_temp': ('RdYlBu_r', 'Temperature [°C]'),
    'salinity':   ('viridis',  'Salinity [PSU]'),
    'water_u':    ('bwr',      'Zonal current [m/s]'),
    'water_v':    ('bwr',      'Meridional current [m/s]'),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (var, (cmap, label)) in zip(axes.flat, variables.items()):
    data = ds[var].isel(depth=0, time=t_idx)
    # Center diverging colormaps at zero for currents
    if 'water_u' in var or 'water_v' in var:
        vmax = float(np.abs(data).max())
        data.plot(ax=ax, cmap=cmap, vmin=-vmax, vmax=vmax,
                  cbar_kwargs={'label': label})
    else:
        data.plot(ax=ax, cmap=cmap, cbar_kwargs={'label': label})
    ax.set_title(var)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

fig.suptitle(f'Surface (0 m) — {str(ds.time.values[t_idx])[:16]}', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Depth profile at a single location

This is what the 1D-CNN "sees" — a vertical slice of T, S, u, v at one grid cell.

The thermocline (sharp temperature drop with depth) is the key physical feature.
A shallow thermocline traps surface heat and amplifies MHW risk.
A deep mixed layer disperses heat and suppresses MHW risk.

In [ ]:
# Centre of the domain
centre_lat = float(ds.lat.values[len(ds.lat)//2])
centre_lon = float(ds.lon.values[len(ds.lon)//2])

profile = ds.sel(lat=centre_lat, lon=centre_lon, method='nearest').isel(time=t_idx)
depths  = ds.depth.values

fig, axes = plt.subplots(1, 4, figsize=(13, 5), sharey=True)

plot_cfg = [
    ('water_temp', 'Temperature [°C]', 'tab:red'),
    ('salinity',   'Salinity [PSU]',   'tab:blue'),
    ('water_u',    'u-current [m/s]',  'tab:green'),
    ('water_v',    'v-current [m/s]',  'tab:purple'),
]

for ax, (var, xlabel, color) in zip(axes, plot_cfg):
    vals = profile[var].values
    ax.plot(vals, depths, 'o-', color=color, markersize=5)
    ax.set_xlabel(xlabel)
    ax.invert_yaxis()          # depth increases downward
    ax.grid(True, alpha=0.3)
    # Mark NaN values (below seafloor)
    nan_depths = depths[np.isnan(vals)]
    if len(nan_depths):
        ax.axhspan(nan_depths[0], depths[-1], alpha=0.1, color='gray', label='below seafloor')
        ax.legend(fontsize=8)

axes[0].set_ylabel('Depth [m]')

fig.suptitle(
    f'Vertical profiles at ({centre_lat:.2f}°N, {centre_lon:.2f}°E)\n'
    f'{str(ds.time.values[t_idx])[:16]}',
    fontsize=12
)
plt.tight_layout()
plt.show()

print(f'Temperature: {float(profile["water_temp"].isel(depth=0)):.1f}°C (surface) → '
      f'{float(profile["water_temp"].dropna("depth").isel(depth=-1)):.1f}°C (deepest valid level)')

## 5. Depth–time heatmap (Hovmöller diagram)

How does temperature evolve with depth over the 3 days?
The diurnal cycle (warming during day, cooling at night) is visible in the surface layers.
Deeper layers are insulated and barely change.

In [ ]:
temp_profile_ts = ds['water_temp'].sel(
    lat=centre_lat, lon=centre_lon, method='nearest'
)  # (time=24, depth=11)

fig, ax = plt.subplots(figsize=(10, 5))

# Mask NaN so contourf doesn't fail on below-seafloor levels
data_plot = temp_profile_ts.values  # (time, depth)

time_nums = np.arange(len(ds.time))
time_labels = [str(t)[:13] for t in ds.time.values]

cf = ax.contourf(
    time_nums, depths, data_plot.T,
    levels=20, cmap='RdYlBu_r'
)
plt.colorbar(cf, ax=ax, label='Temperature [°C]')
ax.set_xticks(time_nums[::4])
ax.set_xticklabels([time_labels[i] for i in time_nums[::4]], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Depth [m]')
ax.set_xlabel('Time (3-hourly)')
ax.invert_yaxis()
ax.set_title(f'Temperature Hovmöller — ({centre_lat:.2f}°N, {centre_lon:.2f}°E)')
ax.axhline(y=0, color='k', linewidth=0.5)
plt.tight_layout()
plt.show()

# Diurnal range at surface
surf = temp_profile_ts.isel(depth=0)
print(f'Surface temp range over 3 days: {float(surf.min()):.2f}°C – {float(surf.max()):.2f}°C  '
      f'(diurnal range: {float(surf.max()-surf.min()):.2f}°C)')

## 6. Surface temperature time series — MHW context

Plot the 3-hourly SST at the centre point and overlay the 18°C stress threshold.
Days above this line contribute to the Stress Degree Day (SDD) accumulation.

In [ ]:
sst_ts = ds['water_temp'].sel(
    lat=centre_lat, lon=centre_lon, method='nearest'
).isel(depth=0)  # (time=24,)

fig, ax = plt.subplots(figsize=(10, 4))

time_nums = np.arange(len(ds.time))
sst_vals  = sst_ts.values

ax.plot(time_nums, sst_vals, 'o-', color='tab:red', markersize=4, label='SST (0 m)')
ax.axhline(18.0, color='navy', linestyle='--', linewidth=1.5, label='18°C stress threshold')

# Shade MHW exceedance area
ax.fill_between(
    time_nums, sst_vals, 18.0,
    where=(sst_vals > 18.0),
    alpha=0.25, color='red', label='Thermal stress (SDD contribution)'
)

ax.set_xticks(time_nums[::4])
ax.set_xticklabels(
    [str(ds.time.values[i])[:13] for i in time_nums[::4]],
    rotation=30, ha='right', fontsize=9
)
ax.set_ylabel('Temperature [°C]')
ax.set_title(f'SST time series — ({centre_lat:.2f}°N, {centre_lon:.2f}°E)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# SDD accumulated over these 3 days (naive sum — no consecutive-day filter)
# Real SDD uses compute_mhw_mask() to enforce the ≥5-day rule first.
# With only 3 days of data here, treat this as a raw anomaly sum.
dt_h = 3.0 / 24.0  # each timestep is 3 hours = 3/24 of a day
naive_sdd = float(np.sum(np.maximum(sst_vals - 18.0, 0.0)) * dt_h)
print(f'Naive SDD (3 days, no consecutive-day filter): {naive_sdd:.2f} °C·day')
print('(Real SDD requires ≥5 consecutive days above threshold)')

## 7. Current vectors

Ocean currents drive lateral heat advection — warm water advected into the region amplifies MHW risk.
The quiver plot shows the direction and speed of surface currents.

In [ ]:
u = ds['water_u'].isel(depth=0, time=t_idx).values   # (lat, lon)
v = ds['water_v'].isel(depth=0, time=t_idx).values
lons_2d, lats_2d = np.meshgrid(ds.lon.values, ds.lat.values)

speed = np.sqrt(u**2 + v**2)

fig, ax = plt.subplots(figsize=(7, 5))

# Background: current speed
pc = ax.pcolormesh(ds.lon.values, ds.lat.values, speed, cmap='YlOrRd', shading='auto')
plt.colorbar(pc, ax=ax, label='Current speed [m/s]')

# Subsample arrows so they don't overlap
step = 2
ax.quiver(
    lons_2d[::step, ::step], lats_2d[::step, ::step],
    u[::step, ::step], v[::step, ::step],
    scale=3.0, width=0.004, color='k', alpha=0.8
)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Surface currents (0 m) — {str(ds.time.values[t_idx])[:16]}')
plt.tight_layout()
plt.show()

print(f'Max surface speed: {speed.max():.3f} m/s')
print(f'Mean surface speed: {speed.mean():.3f} m/s')

## 8. Depth profile at every grid cell — temperature

How uniform is the water column across the region?
Plot all 26×13 = 338 profiles as faint lines, plus the spatial mean.

In [ ]:
temp_snap = ds['water_temp'].isel(time=t_idx)  # (depth=11, lat=26, lon=13)

fig, ax = plt.subplots(figsize=(6, 6))

# Plot every profile as a faint line
for i_lat in range(len(ds.lat)):
    for i_lon in range(len(ds.lon)):
        vals = temp_snap.isel(lat=i_lat, lon=i_lon).values
        ax.plot(vals, depths, color='steelblue', alpha=0.12, linewidth=0.8)

# Spatial mean profile
mean_profile = temp_snap.mean(['lat', 'lon']).values
ax.plot(mean_profile, depths, 'k-', linewidth=2.5, label='Spatial mean')

# 18°C stress threshold
ax.axvline(18.0, color='red', linestyle='--', linewidth=1.5, label='18°C threshold')

ax.invert_yaxis()
ax.set_xlabel('Temperature [°C]')
ax.set_ylabel('Depth [m]')
ax.set_title(f'All 338 T profiles — {str(ds.time.values[t_idx])[:16]}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Surface temperature across all grid cells:')
surf_all = temp_snap.isel(depth=0).values.flatten()
print(f'  min: {surf_all.min():.2f}°C  max: {surf_all.max():.2f}°C  std: {surf_all.std():.2f}°C')

## 9. Salinity–temperature diagram (T-S plot)

Classic oceanographic diagnostic. Each point is one (depth, lat, lon) observation.
Colour = depth. The curve traces water masses — different water masses cluster in distinct regions.

In [ ]:
temp_all = ds['water_temp'].isel(time=t_idx).values  # (depth, lat, lon)
sal_all  = ds['salinity'].isel(time=t_idx).values
dep_all  = np.broadcast_to(
    depths[:, np.newaxis, np.newaxis], temp_all.shape
)

# Flatten, remove NaN
mask  = ~np.isnan(temp_all) & ~np.isnan(sal_all)
T_pts = temp_all[mask]
S_pts = sal_all[mask]
D_pts = dep_all[mask]

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(S_pts, T_pts, c=D_pts, cmap='plasma_r', s=8, alpha=0.6)
plt.colorbar(sc, ax=ax, label='Depth [m]')

ax.axhline(18.0, color='red', linestyle='--', linewidth=1.2, label='18°C threshold')
ax.set_xlabel('Salinity [PSU]')
ax.set_ylabel('Temperature [°C]')
ax.set_title('T-S diagram — all grid cells and depths')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Salinity range: {S_pts.min():.2f} – {S_pts.max():.2f} PSU')
print(f'Temperature range: {T_pts.min():.2f} – {T_pts.max():.2f} °C')

## 10. xarray operations — patterns used in the pipeline

These are the xarray patterns used throughout `src/`. Run them to get comfortable.

In [ ]:
# --- Resample to daily mean (3-hourly → daily) ---
ds_daily = ds.resample(time='1D').mean()
print('After daily resample:', dict(ds_daily.dims))
print('Daily times:', ds_daily.time.values)

In [ ]:
# --- Select a single location by coordinate value ---
point = ds['water_temp'].sel(lat=43.5, lon=-70.0, method='nearest')
print('Single location shape:', point.shape)   # (time=24, depth=11)
print('Actual lat/lon used:', 
      float(ds.lat.sel(lat=43.5, method='nearest')), 
      float(ds.lon.sel(lon=-70.0, method='nearest')))

In [ ]:
# --- Select by integer index (isel) vs label (sel) ---
surface_by_index = ds['water_temp'].isel(depth=0)   # depth index 0 = 0 m
surface_by_label = ds['water_temp'].sel(depth=0)    # depth label 0 m
print('isel(depth=0) == sel(depth=0):', bool((surface_by_index == surface_by_label).all()))

In [ ]:
# --- Group by day-of-year (used in compute_climatology) ---
sst = ds['water_temp'].isel(depth=0)               # (time=24, lat=26, lon=13)
by_doy = sst.groupby('time.dayofyear').mean()
print('After groupby dayofyear:', dict(by_doy.dims))
print('Day-of-year values present:', by_doy.dayofyear.values)

In [ ]:
# --- Extract a numpy array for PyTorch (what HYCOMProxyDataset does) ---
import torch

# Seasonal mean depth profile at centre cell → shape (depth=11, channels=4)
HYCOM_VARIABLES = ['water_temp', 'salinity', 'water_u', 'water_v']

profile_np = np.stack(
    [ds[v].sel(lat=centre_lat, lon=centre_lon, method='nearest').mean('time').values
     for v in HYCOM_VARIABLES],
    axis=-1
).astype(np.float32)  # (11, 4)

profile_t = torch.tensor(profile_np)  # ready for CNN1dEncoder
print('Profile tensor shape:', profile_t.shape)   # (11, 4)
print('Has NaN:', profile_t.isnan().any().item())
print()
print('Channels: [water_temp, salinity, water_u, water_v]')
print('Surface values (depth=0):', profile_t[0].numpy().round(3))

---
## Summary

What you've seen:

| Feature | Key observation |
|---|---|
| Surface temp | ~19–22°C in Aug 2019 in this Gulf of Maine tile — above the 18°C salmon stress threshold |
| Depth profile | Sharp thermocline at ~10–30 m — warm surface layer over cold deep water |
| Diurnal cycle | ~0.3–0.5°C surface warming/cooling over 24 h — captured in 3-hourly data |
| Currents | Structured mesoscale flow; Gulf of Maine circulation visible |
| T-S diagram | Two water masses: warm/low-salinity surface + cold/high-salinity deep |

**What the pipeline does with this:**
1. `HYCOMLoader` fetches 90-day windows → depth profiles as shown in Section 8
2. `compute_mhw_mask()` flags days where SST > 18°C for ≥5 consecutive days
3. `accumulate_sdd()` sums the temperature excess on flagged days → one SDD scalar per location
4. `MHWRiskModel` — the 1D-CNN encodes the vertical profiles (Section 4); Transformer encodes the SST time series (Section 6)
5. `compute_svar()` takes 64 ensemble SDD values → SVaR_95
6. `compute_payout()` maps SVaR_95 → USD payout